In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))


import pandas as pd

df = pd.read_csv('../data/raw/data.csv')

In [5]:
from src.feature_engineering.aggregation import Aggregator


aggregate = Aggregator()
customer_features = aggregate.fit_transform(df)
customer_features.head(10)


,CustomerId,total_transaction_amount,avg_transaction_amount,transaction_count,std_transaction_amount,product,provider,channel
0,CustomerId_1,-10000.0,-10000.000000,1,NaN,airtime,ProviderId_4,ChannelId_2
1,CustomerId_10,-10000.0,-10000.000000,1,NaN,airtime,ProviderId_4,ChannelId_2
2,CustomerId_1001,20000.0,4000.000000,5,6558.963333,financial_services,ProviderId_4,ChannelId_3
3,CustomerId_1002,4225.0,384.090909,11,560.498966,financial_services,ProviderId_4,ChannelId_2
4,CustomerId_1003,20000.0,3333.333333,6,6030.478146,airtime,ProviderId_6,ChannelId_3
5,CustomerId_1004,2000.0,2000.000000,1,NaN,airtime,ProviderId_1,ChannelId_3
6,CustomerId_1005,48800.0,5422.222222,9,5983.681513,financial_services,ProviderId_4,ChannelId_3
7,CustomerId_1006,1000.0,1000.000000,1,NaN,financial_services,ProviderId_5,ChannelId_3
8,CustomerId_1007,28000.0,28000.000000,1,NaN,financial_services,ProviderId_5,ChannelId_3
9,CustomerId_1008,20000.0,10000.000000,2,0.000000,financial_services,ProviderId_3,ChannelId_3


In [2]:
from src.feature_engineering.time_feature import TimeExtractor

time_extractor = TimeExtractor()

df_time = time_extractor.fit_transform(df)

df_time[[
    "TransactionStartTime",
    "transaction_hour",
    "transaction_day",
    "transaction_month",
    "transaction_year"
]].head(10)

,TransactionStartTime,transaction_hour,transaction_day,transaction_month,transaction_year
0,2018-11-15 02:18:49+00:00,2,15,11,2018
1,2018-11-15 02:19:08+00:00,2,15,11,2018
2,2018-11-15 02:44:21+00:00,2,15,11,2018
3,2018-11-15 03:32:55+00:00,3,15,11,2018
4,2018-11-15 03:34:21+00:00,3,15,11,2018
5,2018-11-15 03:35:10+00:00,3,15,11,2018
6,2018-11-15 03:44:31+00:00,3,15,11,2018
7,2018-11-15 03:45:13+00:00,3,15,11,2018
8,2018-11-15 04:14:59+00:00,4,15,11,2018
9,2018-11-15 04:31:48+00:00,4,15,11,2018


In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

hot_cols = ['product']
ordinal_cols = ['provider', 'channel']

preprocessor = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(handle_unknown='ignore'), hot_cols),
        ("label", OrdinalEncoder(), ordinal_cols)
    ],
    remainder="passthrough"
)

X = preprocessor.fit_transform(customer_features)


In [19]:
onehot_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(hot_cols)

ordinal_feature_names = ordinal_cols

numeric_cols = [col for col in customer_features.columns if col not in hot_cols + ordinal_cols]

all_feature_names = list(onehot_feature_names) + ordinal_feature_names + numeric_cols

X_df = pd.DataFrame(X, columns=all_feature_names)

X_df['CustomerId'] = customer_features['CustomerId'].values



In [20]:
X_df.head(10)

,product_airtime,product_data_bundles,product_financial_services,product_movies,product_other,product_ticket,product_transport,product_tv,product_utility_bill,provider,channel,CustomerId,total_transaction_amount,avg_transaction_amount,transaction_count,std_transaction_amount
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,CustomerId_1,-10000.0,-10000.0,1,NaN
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,CustomerId_10,-10000.0,-10000.0,1,NaN
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,2.0,CustomerId_1001,20000.0,4000.0,5,6558.963333
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,CustomerId_1002,4225.0,384.090909,11,560.498966
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,2.0,CustomerId_1003,20000.0,3333.333333,6,6030.478146
5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,CustomerId_1004,2000.0,2000.0,1,NaN
6,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,2.0,CustomerId_1005,48800.0,5422.222222,9,5983.681513
7,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,2.0,CustomerId_1006,1000.0,1000.0,1,NaN
8,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,2.0,CustomerId_1007,28000.0,28000.0,1,NaN
9,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,CustomerId_1008,20000.0,10000.0,2,0.0


In [23]:
X_df.isnull().sum()

product_airtime               0
product_data_bundles          0
product_financial_services    0
product_movies                0
product_other                 0
product_ticket                0
product_transport             0
product_tv                    0
product_utility_bill          0
provider                      0
channel                       0
CustomerId                    0
total_transaction_amount      0
avg_transaction_amount        0
transaction_count             0
std_transaction_amount        0
dtype: int64

In [ ]:
X_df['std_transaction_amount'].fillna(0, inplace=True)

In [ ]:
numeric_cols = [
    'total_transaction_amount',
    'avg_transaction_amount',
    'transaction_count',
    'std_transaction_amount'
]



In [29]:
from sklearn.preprocessing import StandardScaler

numeric_cols = [
    'total_transaction_amount',
    'avg_transaction_amount',
    'transaction_count',
    'std_transaction_amount'
]

scaler = StandardScaler()
X_df[numeric_cols] = scaler.fit_transform(X_df[numeric_cols])

X_df[numeric_cols].describe()


,total_transaction_amount,avg_transaction_amount,transaction_count,std_transaction_amount
count,3.742000e+03,3742.000000,3.742000e+03,3.742000e+03
mean,2.373539e-18,0.000000,-7.595326e-18,4.747079e-18
std,1.000134e+00,1.000134,1.000134e+00,1.000134e+00
min,-3.867280e+01,-2.628365,-2.534591e-01,-1.404325e-01
25%,-6.170920e-02,-0.087762,-2.431409e-01,-1.352597e-01
50%,-5.584873e-02,-0.078316,-1.915502e-01,-1.075580e-01
75%,-3.377694e-02,-0.064636,-5.741433e-02,-7.080680e-02
max,3.065193e+01,51.206316,4.194775e+01,3.402451e+01
